# High-Q silicon resonator in BeamZ

This notebook is a BeamZ-native implementation of Flexcompute's **[Tidy3D High-Q silicon resonator notebook](https://www.flexcompute.com/tidy3d/examples/notebooks/HighQSi/)**, explicitly our reference to replicate. The reference reproduces the transmission study of Zhang *et al.*, [*Optics Letters* 43, 1842–1845 (2018)](https://www.osapublishing.org/ol/abstract.cfm?uri=ol-43-8-1842), using a periodic silicon unit cell with symmetric and symmetry-broken resonators. It was originally developed by Romil Audhkhasi (USC).

The BeamZ workflow retains the 650 nm square period, silica substrate, paired 260 nm-thick silicon bars, 80 nm gap, $\delta=0$ and $\delta=20$ nm cases, broadband transmission monitor, normalization run, and periodic $x/y$ boundaries with CPML along $z$. A very wide Gaussian beam spanning the unit cell is used as BeamZ's current approximation to a uniform normally incident plane wave.

High-Q lines require long time signals, fine spectral sampling, and adequate resolution inside silicon. The full configuration follows the reference with 1000 wavelengths, 7 ps resonator runs, and at least 32 cells per shortest material wavelength while capping the background spacing at $P/32$. `BEAMZ_DOCS_TEST=1` uses a coarse uniform 8-cells-per-period grid, 21 monitor wavelengths, and short runtimes only for automated CPU smoke testing.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
try:
    from IPython.display import display
except ImportError:
    display = print

import beamz as bz

test_mode = os.environ.get("BEAMZ_DOCS_TEST") == "1"
# Periodic boundaries currently execute through BeamZ's JAX kernels.
execution_backend = "jax"
nm = bz.nm
um = bz.um


## 1. Spectrum, materials, and geometry

The wavelength range and nondispersive indices match the reference. The parameter $\delta=w_1-w_2$ breaks the in-plane symmetry while keeping $w_1+w_2=400$ nm. This converts a symmetry-protected state into a narrow radiative resonance.

In [ ]:
num_frequencies = 21 if test_mode else 1000
wavelengths = np.linspace(1050, 1400, num_frequencies) * nm
freqs = bz.LIGHT_SPEED / wavelengths
freq0 = freqs[len(freqs) // 2]
fwidth = freqs[0] - freqs[-1]

n_sio2 = 1.46
n_si = 3.52
mat_air = bz.Material(permittivity=1.0)
mat_sio2 = bz.Material(permittivity=n_sio2**2)
mat_si = bz.Material(permittivity=n_si**2)

spacing = 1.5 * um
period_x = period_y = period = 650 * nm
t_si = 260 * nm
gap = 80 * nm
bar_length = 480 * nm
width_sum = 400 * nm
Lz = 2 * spacing + 2 * t_si
dl = period / (8 if test_mode else 32)


def bar_widths(delta):
    w1 = 0.5 * (width_sum + delta)
    w2 = width_sum - w1
    return w1, w2


## 2. Parameterized periodic unit cell

`bz.Periodic(axes=("x", "y"))` couples each pair of opposite Yee-lattice faces with zero phase, which is the normal-incidence condition used by the reference. CPML remains independent on the front and back faces. In full mode, BeamZ's material-aware policy keeps every cell no larger than $P/32$ and resolves the shortest in-material silicon wavelength with at least 32 cells. Because the current Gaussian-beam operator requires an isotropic grid, that conservative silicon spacing applies throughout this model and is finer in $x/y$ than the reference.

In [ ]:
def make_design(delta=None):
    # Omitting explicit dimensions marks this as centered public geometry;
    # make_sim supplies the actual periodic-cell size.
    design = bz.Design(background=mat_air)
    # The substrate occupies the lower half-space and is clipped by the domain.
    design += bz.Box(
        center=(0, 0, -Lz / 2),
        size=(period_x, period_y, 2 * (spacing + t_si)),
        material=mat_sio2,
    )
    if delta is None:
        return design

    w1, w2 = bar_widths(delta)
    center_y = 0.5 * (w1 - w2)
    design += bz.Box(
        center=(0, center_y + 0.5 * (gap + w1), 0.5 * t_si),
        size=(bar_length, w1, t_si), material=mat_si,
    )
    design += bz.Box(
        center=(0, center_y - 0.5 * (gap + w2), 0.5 * t_si),
        size=(bar_length, w2, t_si), material=mat_si,
    )
    return design


def make_sim(delta=None, *, normalization=False):
    pulse = bz.GaussianPulse(freq0=freq0, fwidth=fwidth)
    source = bz.GaussianBeamSource(
        center=(0, 0, Lz / 2 - spacing + 2 * dl),
        size=(period_x, period_y, 0),
        source_time=pulse,
        direction="-z",
        pol_angle=0.0,
        waist_radius=10 * period,
        wavelength=bz.LIGHT_SPEED / freq0,
        power=1.0,
    )
    flux = bz.FluxMonitor(
        center=(0, 0, -Lz / 2 + spacing - 2 * dl),
        size=(period_x, period_y, 0),
        freqs=freqs,
        name="flux",
    )
    full_runtime = 7e-12
    runtime = (20 / freq0 if test_mode else full_runtime)
    if normalization:
        runtime = (10 / freq0 if test_mode else full_runtime / 10)
    grid_spec = (
        bz.GridSpec.uniform(dl)
        if test_mode
        else bz.GridSpec.auto(
            wavelength=float(wavelengths.min()),
            min_steps_per_wvl=32,
            dl_max=dl,
        )
    )
    return bz.Simulation(
        design=make_design(None if normalization else delta),
        size=(period_x, period_y, Lz),
        sources=[source],
        monitors=[flux],
        boundaries=[
            bz.Periodic(axes=("x", "y")),
            bz.PML(
                edges=("front", "back"),
                thickness=12 * dl,
                formulation="cpml",
            ),
        ],
        grid_spec=grid_spec,
        run_time=runtime,
    )


## 3. Define and inspect the three cases

The normalization cell contains only the air–silica interface. The two resonator cells use $\delta=0$ and 20 nm. Setup construction and plotting do not advance any FDTD timesteps.

In [ ]:
sim_empty = make_sim(normalization=True)
sim_delta0 = make_sim(0 * nm)
sim_delta20 = make_sim(20 * nm)

print(f"unit-cell grid = {sim_delta0.grid.shape}")
print(f"normalization steps = {sim_empty.num_steps:,}")
print(f"resonator steps = {sim_delta0.num_steps:,}")
display(sim_delta0)

fig, axes = sim_delta0.plot(
    z=0.5 * t_si, y=gap, figsize=(10, 4.2),
    source_markers=True, monitor_markers=True, show=False,
)
for axis in np.asarray(axes).flat:
    axis.grid(False)
fig.suptitle("Symmetric periodic silicon resonator", y=1.02)
plt.show()


## 4. Run the three simulations

Keep this execution cell separate so the geometry, periodic boundaries, grid, and runtimes can be reviewed first. The long 7 ps runs are necessary to resolve the high-Q response; they have not been executed in this checked-in notebook.

In [ ]:
simulations = {
    "normalization": sim_empty,
    "Si-resonator-delta-0": sim_delta0,
    "Si-resonator-delta-20": sim_delta20,
}
results = {
    name: simulation.run(progress=not test_mode, backend=execution_backend)
    for name, simulation in simulations.items()
}


## 5. Normalize and compare transmission

The empty-cell flux captures the air–silica interface and the identical source/monitor discretization. Dividing the two resonator spectra by that baseline isolates the resonator response, following the reference. The symmetric structure supports a symmetry-protected state; the 20 nm asymmetry opens radiative coupling and produces a narrow Fano feature.

In [ ]:
flux_norm = np.asarray(results["normalization"]["flux"].flux, dtype=float)
flux_delta0 = np.asarray(results["Si-resonator-delta-0"]["flux"].flux, dtype=float)
flux_delta20 = np.asarray(results["Si-resonator-delta-20"]["flux"].flux, dtype=float)
denominator = np.where(np.abs(flux_norm) > 1e-15, flux_norm, np.nan)
trans_delta0 = flux_delta0 / denominator
trans_delta20 = flux_delta20 / denominator

fig, ax = plt.subplots(figsize=(6.2, 4.5))
ax.plot(wavelengths / nm, trans_delta0, color="red", label=r"$\delta=0$")
ax.plot(wavelengths / nm, trans_delta20, color="blue", label=r"$\delta=20\,\mathrm{nm}$")
ax.set(xlabel="wavelength (nm)", ylabel="normalized transmission", xlim=(1050, 1400), ylim=(0, 1))
ax.grid(alpha=0.25)
ax.legend()
plt.show()


## Interpretation and convergence

Compare the full-resolution spectrum with the published and Tidy3D curves by resonance position, linewidth, and Fano line shape—not only the depth of one sampled point. A converged result should refine the 32-cells-per-period grid, extend the time window until the resonant field has decayed sufficiently, and increase spectral sampling around the narrow feature. The reduced CPU mode is only a workflow check and cannot resolve a high Q factor.

The reference also points to the related [Tidy3D germanium Fano metasurface example](https://www.flexcompute.com/tidy3d/examples/notebooks/HighQGe/) for another periodic high-Q workflow.